# 11 — Evaluate VAE

Load a checkpoint, run eval on the validation split, and visualise mid-slices.

In [ ]:
import yaml
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from poregen.dataset.loader import PatchDataset
from poregen.models.vae import build_vae
from poregen.losses import compute_total_loss
from poregen.metrics import mse, mae, psnr, sharpness_proxy, segmentation_metrics, latent_stats
from poregen.training import select_device, get_autocast_dtype, load_checkpoint

## Config + Checkpoint

In [ ]:
CFG_PATH = Path("../src/poregen/configs/example_vae.yaml")
cfg = yaml.safe_load(CFG_PATH.read_text())

EXP_NAME = "conv_baseline"
CKPT_PATH = Path(f"../runs/vae/{EXP_NAME}/last.ckpt")

device = select_device()
autocast_dtype = get_autocast_dtype(device)

## Load model

In [ ]:
model = build_vae(
    cfg["model"]["name"],
    z_channels=cfg["model"]["z_channels"],
    base_channels=cfg["model"]["base_channels"],
    n_blocks=cfg["model"]["n_blocks"],
    patch_size=cfg["model"]["patch_size"],
).to(device)

step, meta = load_checkpoint(CKPT_PATH, model, map_location=device)
print(f"Loaded checkpoint at step {step}")

## Load validation data

In [ ]:
DATA_ROOT = Path("../data") / cfg["data"].get("dataset_root", "split_v1")
val_ds = PatchDataset(DATA_ROOT / "patch_index.parquet", DATA_ROOT, split="val")
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f"Val patches: {len(val_ds):,}")

## Run eval

In [ ]:
from poregen.training.engine import eval_step

loss_cfg = cfg["loss"]
loss_fn = lambda output, batch, step: compute_total_loss(output, batch, step, loss_cfg)

all_losses = []
all_seg = []
all_latent = []
outputs_for_viz = []
batches_for_viz = []

model.eval()
for batch in val_loader:
    losses, output = eval_step(model, batch, loss_fn, step=0, device=device, autocast_dtype=autocast_dtype)
    all_losses.append(losses)
    
    # Metrics
    with torch.no_grad():
        all_seg.append(segmentation_metrics(output.mask_logits, batch["mask"].to(device)))
        all_latent.append(latent_stats(output.mu, output.logvar))
    
    if len(outputs_for_viz) < 2:
        outputs_for_viz.append(output)
        batches_for_viz.append(batch)

# Aggregate
import numpy as np
mean_loss = {k: np.mean([l[k] for l in all_losses]) for k in all_losses[0]}
mean_seg = {k: np.mean([s[k] for s in all_seg]) for k in all_seg[0]}
mean_latent = {k: np.mean([l[k] for l in all_latent]) for k in all_latent[0]}

print("=== Losses ===")
for k, v in mean_loss.items():
    print(f"  {k:20s}: {v:.4f}")

print("\n=== Segmentation ===")
for k, v in mean_seg.items():
    print(f"  {k:20s}: {v:.4f}")

print("\n=== Latent ===")
for k, v in mean_latent.items():
    print(f"  {k:20s}: {v:.4f}")

## Visualise mid-slices

In [ ]:
def show_midslice(batch, output, sample_idx=0):
    """Show XCT and mask: GT vs reconstruction (mid-Z slice)."""
    mid = batch["xct"].shape[2] // 2
    
    gt_xct  = batch["xct"][sample_idx, 0, mid].cpu().numpy()
    gt_mask = batch["mask"][sample_idx, 0, mid].cpu().numpy()
    
    pred_xct  = torch.sigmoid(output.xct_logits[sample_idx, 0, mid]).detach().cpu().numpy()
    pred_mask = (torch.sigmoid(output.mask_logits[sample_idx, 0, mid]) > 0.5).float().detach().cpu().numpy()
    
    fig, axes = plt.subplots(2, 2, figsize=(8, 8))
    axes[0, 0].imshow(gt_xct, cmap="gray"); axes[0, 0].set_title("GT XCT")
    axes[0, 1].imshow(pred_xct, cmap="gray"); axes[0, 1].set_title("Recon XCT")
    axes[1, 0].imshow(gt_mask, cmap="gray"); axes[1, 0].set_title("GT Mask")
    axes[1, 1].imshow(pred_mask, cmap="gray"); axes[1, 1].set_title("Recon Mask")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

for i, (batch, output) in enumerate(zip(batches_for_viz, outputs_for_viz)):
    print(f"\n--- Batch {i} ---")
    show_midslice(batch, output)